# <span style="color:red"> .loc, .iloc, and indexes </span>

<font size = "3">

This supplementary notebook is intended to clear up some confusion on `.loc` and `.iloc`, which requires us to know something about indexes of DataFrames.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

carfeatures = pd.read_csv('data_raw/subfolder1/annoying_subfolder/features.csv')

<font size = "3">

Let's display the carfeatures DataFrame and also print out its .columns attribute, and the length of the .columns attribute.

In [ ]:
display(carfeatures)

print()
print("carfeatures.columns is:")
print(carfeatures.columns)
print()
print("The length of carfeatures.columns is:")
print(len(carfeatures.columns))

<font size = "3">

At first glance, you might think that there are 8 columns in the DataFrame, with the first column having the row numbers 0, 1, 2, 3, ... However, this is not a "column" as far as Pandas is concerned. It is an **index**.

The index does not have to be 0, 1, 2, 3, ... In fact, it often makes sense to use one of the variables as an index if it can be used as a **unique** identifier for each row.

For example, we can choose the "vehicle_id" column as our index:

In [ ]:
# Just a quick check that the values in the vehicle_id column are unique

unique_ids = carfeatures["vehicle_id"].nunique()
print("There are", unique_ids, "unique vehicle_id numbers")
print("Matches number of rows:", len(carfeatures) == unique_ids)

# Create a copy of the DataFrame where vehicle_id is the index
carfeatures_2 = carfeatures.set_index("vehicle_id")

display(carfeatures_2)
print("carfeatures_2.columns is:")
print(carfeatures_2.columns)

<font size = "3">

Note that "vehicle_id" is no longer considered a column of the dataset. It is the index:

In [ ]:
print(carfeatures_2.index)

<font size = "3">

It's important to note that the layout of the original DataFrame is unchanged:

In [ ]:
display(carfeatures.head())

<font size = "3">

Let's grab the value 18.0 from the upper left corner of "carfeatures_2" using `.iloc`:

In [ ]:
display(carfeatures_2.head())

upper_left_val = carfeatures_2.iloc[0, 0]

print("upper left value:", upper_left_val)

<font size = "3">

We can grab the same value with `.loc`, but we must use the name of the column and the name corresponding to the index:

In [ ]:
display(carfeatures_2.head())

upper_left_val = carfeatures_2.loc["C-1689780", "mpg"]

print("upper left value:", upper_left_val)

<font size = "3">

To grab the weight of car "A-1689813", we can use `.loc` like so:

In [ ]:
car_weight = carfeatures_2.loc["A-1689813", "weight"]
print(car_weight)

<font size = "4">

Or, noting that this value is in row 3 and column 4 (counting from zero), we can use `.iloc` instead:

In [ ]:
car_weight = carfeatures_2.iloc[3, 4]
print(car_weight)

<font size = "3">

**Resetting the index:** We can move the index column back to a normal column by using the `.reset_index` method:

In [ ]:
carfeatures_3 = carfeatures_2.reset_index()

display(carfeatures_3.head())

<font size = "3">

But "carfeatures_2" is unchanged:

In [ ]:
display(carfeatures_2.head())

<font size = "3">

We now have 3 copies of the DataFrame floating around, which is a total waste of memory. Of course, we could continually overwrite the same variable to save space:


In [ ]:
carfeatures = pd.read_csv('data_raw/subfolder1/annoying_subfolder/features.csv')

display(carfeatures.head())
carfeatures = carfeatures.set_index("vehicle_id")
display(carfeatures.head())
carfeatures = carfeatures.reset_index()
display(carfeatures.head())

<font size = "3">

For truly large datasets, this is still not optimal. For a few milliseconds, before the assignment operation completes, there is still a duplicate DataFrame temporarily created. For massive datasets, this temporary copy might make your computer run out of memory! Instead, we can set the "inplace" argument equal to `True`.

In [ ]:
carfeatures = pd.read_csv('data_raw/subfolder1/annoying_subfolder/features.csv')

display(carfeatures.head())
carfeatures.set_index("vehicle_id", inplace=True)
display(carfeatures.head())
carfeatures.reset_index(inplace=True)
display(carfeatures.head())

<font size = "3">

We can also manually assign different numbers to a DataFrame's index:

In [ ]:
carfeatures = pd.read_csv('data_raw/subfolder1/annoying_subfolder/features.csv')

carfeatures.index = pd.RangeIndex(start = 30, stop = 428)

display(carfeatures)

<font size = "3">

If we use `.loc`, we must use the new numbering we've chosen!

In [ ]:
try:
    upper_left = carfeatures.loc[0, "mpg"]
except KeyError:
    print(".loc cannot find a row labeled '0' !")

upper_left = carfeatures.loc[30, "mpg"]
print(upper_left)

# Study summary: `.loc`, `.iloc`, and indexes

**Purpose:** distinguish an observation's current position from its row label. They happen to coincide in a fresh default-index CSV, but they do not mean the same thing.

| Tool | Meaning | Example |
|---|---|---|
| `.columns` | Labels of ordinary data columns | `df.columns` |
| `.index` | Labels identifying rows | `df.index` |
| `.iloc[row, col]` | Select by integer position | `df.iloc[0, 0]` |
| `.loc[row_label, col_label]` | Select by labels | `df.loc["C-1689780", "mpg"]` |
| `.set_index` | Make a column the row index | `by_id = df.set_index("vehicle_id")` |
| `.reset_index` | Move the index back into a column | `regular = by_id.reset_index()` |
| `.nunique` | Count distinct nonmissing entries | `df["vehicle_id"].nunique()` |
| `pd.RangeIndex` | Construct integer row labels | `pd.RangeIndex(30, 30 + len(df))` |

### Useful code block: retrieve a car by its identifier

```python
assert df["vehicle_id"].nunique() == len(df)
by_id = df.set_index("vehicle_id")
value_by_position = by_id.iloc[0, 0]
value_by_label = by_id.loc["C-1689780", "mpg"]
```

In this notebook those two lookups retrieve the same first car's mpg. The assertion checks the supplied dataset's identifiers for uniqueness. A general identifier check would also check that none are missing.

By default, `set_index` removes `vehicle_id` from the ordinary columns. It remains available as the index. The original `df` remains unchanged when you assign the result to `by_id`.

### Useful code block: demonstrate that labels are not positions

```python
shifted = df.copy()
shifted.index = pd.RangeIndex(30, 30 + len(shifted))
first_row = shifted.iloc[0]           # first row by position
same_row = shifted.loc[30]           # first row has label 30
# shifted.loc[0] raises KeyError: there is no row label 0
```

An assigned index must have exactly as many labels as there are rows. Changing labels does not change the data's row order.

### Slices, dimensions, and resetting

`.iloc[1:6]` selects positions 1–5; stop 6 is excluded. `.loc[1:6]` is a label slice and ordinarily includes label 6 when present in a valid slice. With an identifier index, use identifier labels in `.loc`. After sorting, retained labels identify old rows while `.iloc` follows the new order.

`df.iloc[0]` usually returns a Series for one row; `df.iloc[[0]]` returns a one-row DataFrame. Selecting both a single row and a single column yields a scalar for a unique match. Duplicate row labels can make a `.loc` lookup return several observations.

```python
by_id = by_id.reset_index()                 # keep old IDs as a column
fresh_labels = filtered.reset_index(drop=True)  # discard the old index
```

`inplace=True` changes the receiving DataFrame and returns `None`. Otherwise, keep the returned result. Do not assume `inplace=True` means an operation allocates no temporary memory.

**Quiz extension:** if asked for the five highest-scoring rows, use `sort_values(...).iloc[:5]`, not `.loc[:4]`. The latter refers to labels, which sorting does not necessarily renumber.
